# REMNET Stage 3 — Section 1 ONLY | Automatic Visual-Semantic Compatibility Audit (Kaggle CPU)

**Purpose:** Perform the requested Section 1 work **in Kaggle**, not in this chat. The notebook automatically obtains BOTH public datasets, counts all 10 Garbage Classification V2 classes, restricts the experiment to the 6 REMNET classes, and independently compares the image content of these classes—particularly **all external Trash images**—with the original public reference dataset using a **pretrained CLIP vision–language model**.

**Sources:**
- Original: https://www.kaggle.com/datasets/saumyamohandas/garbage-classification-image-dataset
- External (requested version 12): https://www.kaggle.com/datasets/sumn2u/garbage-classification-v2

**CPU only:** no CUDA, no REMNET checkpoints, no retraining, no CodeCarbon, no pHash, and no duplicate exclusion. CLIP is used **only for an independent class-definition/semantic audit**, NOT to score REMNET's classification performance or choose checkpoints. The full 7,463 images are counted, but the image-content audit uses seeded reference samples for five classes and **every Trash image** in the external dataset. This distinction is reported.

**Evidentiary boundary:** CLIP scores are a useful automatic screen, **not verified ground truth**. A computer-vision model cannot certify every class assignment or settle ambiguous Trash labels on its own. The notebook produces `AUTO_SUPPORTED` or `REVIEW_REQUIRED` with reasons. It does **not** falsely claim that an uninspected human review has occurred. Outputs are written to `/kaggle/working/REMNET_Stage3_Section1_Both_Datasets/`.

**Dependencies:** KaggleHub for datasets; Hugging Face Transformers for a pretrained CLIP model. The model weights require network access once unless already cached. If Kaggle blocks the model download, the notebook records a clearly identified blocked audit rather than claiming successful semantic verification.

In [3]:
# 1 — Configuration and CPU-only imports
from pathlib import Path
from collections import OrderedDict
from PIL import Image, ImageOps, ImageDraw, ImageFont
import os, sys, subprocess, importlib, csv, json, math, random, datetime, zipfile

EXTERNAL_DATASET_HANDLE = 'sumn2u/garbage-classification-v2'
EXTERNAL_DATASET_VERSION = '12'      # version reported by your earlier Kaggle run
ORIGINAL_DATASET_HANDLE = 'saumyamohandas/garbage-classification-image-dataset'

# Known mount locations; these are only cache hints. The notebook WILL auto-download
# BOTH datasets with KaggleHub if these directories are absent.
EXTERNAL_DATASET_PATHS = [
    '/kaggle/input/datasets/sumn2u/garbage-classification-v2',
    '/kaggle/input/garbage-classification-v2',
]
ORIGINAL_DATASET_PATHS = [
    '/kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset',
    '/kaggle/input/garbage-classification-image-dataset',
]
KEEP = ('cardboard','glass','metal','paper','plastic','trash')
EXCLUDE = ('battery','biological','clothes','shoes')
ALL = tuple(sorted(KEEP + EXCLUDE))
EXTS = {'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff'}
OUTPUT = Path('/kaggle/working/REMNET_Stage3_Section1_Both_Datasets') if Path('/kaggle').exists() else Path.cwd()/'REMNET_Stage3_Section1_Both_Datasets'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Section 1: Auto-download BOTH Kaggle datasets | CPU only')
print('Original dataset:', ORIGINAL_DATASET_HANDLE)
print('External dataset:', EXTERNAL_DATASET_HANDLE, '| requested version:', EXTERNAL_DATASET_VERSION)
print('Outputs:', OUTPUT)


Section 1: Auto-download BOTH Kaggle datasets | CPU only
Original dataset: saumyamohandas/garbage-classification-image-dataset
External dataset: sumn2u/garbage-classification-v2 | requested version: 12
Outputs: /kaggle/working/REMNET_Stage3_Section1_Both_Datasets


### Automatic dataset loading

The next cell **first searches for a previously mounted copy**. If a dataset is missing, it calls KaggleHub for that dataset. The returned path is used directly; it does **not** assume the download will appear in a hard-coded `/kaggle/input` directory. When a Kaggle dataset contains a ZIP archive rather than class folders, the notebook also detects and extracts that archive safely into `/kaggle/working`.

The KaggleHub API is documented at https://github.com/Kaggle/kagglehub. If the Kaggle service refuses a download or network access is disabled, the notebook prints the underlying error and a practical recovery instruction rather than claiming the dataset was attached.


In [4]:
# 2 — Automatically acquire BOTH datasets; locate their actual class folders

def list_kaggle_mounts():
    base = Path('/kaggle/input')
    if not base.is_dir():
        return []
    return [str(p) for p in base.iterdir() if p.is_dir()]


def local_mount(candidates, handle):
    for name in candidates:
        p = Path(name)
        if p.is_dir():
            return p
    slug = handle.split('/')[-1]
    base = Path('/kaggle/input')
    if base.is_dir():
        for p in sorted(base.iterdir()):
            if p.is_dir() and (p.name.lower() == slug or slug in p.name.lower()):
                return p
        # Older Kaggle input paths can include a datasets/<owner>/<slug> nesting.
        owners = base/'datasets'
        if owners.is_dir():
            for owner in sorted(owners.iterdir()):
                if owner.is_dir() and (owner/slug).is_dir():
                    return owner/slug
    return None


def import_kagglehub():
    try:
        import kagglehub
        return kagglehub
    except ImportError:
        print('Installing the official KaggleHub client...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub>=1.0.2'])
        importlib.invalidate_caches()
        import kagglehub
        return kagglehub


def get_dataset(handle, expected_paths, preferred_version=None):
    mounted = local_mount(expected_paths, handle)
    if mounted is not None:
        print(f'Using existing Kaggle input [{handle}]: {mounted}')
        return mounted, 'already mounted', 'UNVERIFIED (attached revision)'

    kh = import_kagglehub()
    versioned = f'{handle}/versions/{preferred_version}' if preferred_version else handle
    trials = [versioned] if versioned == handle else [versioned, handle]
    errors = []
    for dataset_handle in trials:
        try:
            print(f'Auto-downloading with KaggleHub: {dataset_handle}')
            path = Path(kh.dataset_download(dataset_handle))
            if path.is_dir():
                version_label = preferred_version if dataset_handle == versioned and preferred_version else 'UNVERIFIED (latest fallback)'
                print(f'  SUCCESS: {path}')
                return path, 'kagglehub auto-download', version_label
            errors.append(f'{dataset_handle}: returned missing/non-directory path {path}')
        except Exception as exc:
            msg = f'{dataset_handle}: {type(exc).__name__}: {exc}'
            print('  Download attempt failed:', msg[:800])
            errors.append(msg)

    message = ('Automatic KaggleHub download failed for ' + handle + '\n' +
               '\n'.join(errors) + '\n' +
               'Kaggle recovery: enable Internet in notebook Settings if your environment requires it; '
               'check dataset access/terms; or attach the same public dataset through Add Input. '
               'The download is attempted automatically before this error appears.')
    (OUTPUT / ('Section1_Download_Failure_' + handle.split('/')[-1] + '.txt')).write_text(message,encoding='utf-8')
    raise RuntimeError(message)


def find_class_roots(root, needed):
    needed = set(needed)
    hits = []
    for parent, dirs, _files in os.walk(root):
        found = {d.lower():Path(parent)/d for d in dirs}
        if needed.issubset(found):
            hits.append((Path(parent), found))
            dirs[:] = [d for d in dirs if d.lower() not in needed]
    return hits


def safe_zip_extract(zip_path, target):
    target.mkdir(parents=True,exist_ok=True)
    base = target.resolve()
    with zipfile.ZipFile(zip_path) as z:
        for member in z.infolist():
            dest = (base/member.filename).resolve()
            if not dest.is_relative_to(base):
                raise ValueError(f'Unsafe archive member: {member.filename}')
        z.extractall(base)
    return base


def roots_with_archive_fallback(root, required, label):
    candidates = find_class_roots(root, required)
    if candidates:
        return candidates
    archives = sorted(root.rglob('*.zip'))
    if archives:
        print(f'{label}: class folders not directly visible; extracting {len(archives)} archive(s).')
    for j, archive in enumerate(archives):
        dest = OUTPUT / 'extracted_datasets' / label / f'archive_{j:02d}'
        if not dest.exists() or not any(dest.iterdir()):
            safe_zip_extract(archive,dest)
        candidates += find_class_roots(dest, required)
    return candidates


print('Existing Kaggle inputs:', list_kaggle_mounts())
EXTERNAL_MOUNT, EXTERNAL_LOAD_METHOD, EXTERNAL_VERSION_ACTUAL = get_dataset(
    EXTERNAL_DATASET_HANDLE, EXTERNAL_DATASET_PATHS, EXTERNAL_DATASET_VERSION)
ORIGINAL_MOUNT, ORIGINAL_LOAD_METHOD, ORIGINAL_VERSION_ACTUAL = get_dataset(
    ORIGINAL_DATASET_HANDLE, ORIGINAL_DATASET_PATHS)

external_candidates = roots_with_archive_fallback(EXTERNAL_MOUNT, ALL, 'external_v2')
if not external_candidates:
    raise RuntimeError(f'KaggleHub returned {EXTERNAL_MOUNT}, but no full ten-class folder tree was found. '
                       'Check the saved dataset files and any embedded archive structure.')

def rank_external(candidate):
    path = candidate[0]
    parts = [p.lower() for p in path.parts]
    return (0 if 'original' in parts else 1, 0 if not any('resized' in x or 'standardized' in x for x in parts) else 1,
            len(path.parts),str(path))
external_candidates.sort(key=rank_external)
EXTERNAL_CLASS_ROOT, EXTERNAL_DIRS = external_candidates[0]

original_candidates = roots_with_archive_fallback(ORIGINAL_MOUNT, KEEP, 'original_stage2')
original_directories = {}
for cl in KEEP:
    dirs = list(dict.fromkeys(classdirs[cl] for _parent,classdirs in original_candidates))
    # Use the actual Training, Testing and Validation folders where present.
    split_dirs = [p for p in dirs if any(part.lower() in ('training','testing','validation','train','test','val') for part in p.parts)]
    original_directories[cl] = sorted(split_dirs if split_dirs else dirs[:1],key=str)

ORIGINAL_AVAILABLE = all(original_directories[cl] for cl in KEEP)
print('External image-class root:', EXTERNAL_CLASS_ROOT)
print('Original Stage 2 root:', ORIGINAL_MOUNT)
print('Original split/class folder found for every class:', ORIGINAL_AVAILABLE)
if not ORIGINAL_AVAILABLE:
    print('WARNING: The downloaded original data lacks one or more six-class folder trees. Class comparisons are not yet verified.')

(OUTPUT/'Section1_Input_Diagnostic.txt').write_text(
    f'External handle: {EXTERNAL_DATASET_HANDLE}\nExternal resolved mount: {EXTERNAL_MOUNT}\n'
    f'External download method: {EXTERNAL_LOAD_METHOD}\nRequested V2 version: {EXTERNAL_DATASET_VERSION}\n'
    f'External version evidence: {EXTERNAL_VERSION_ACTUAL}\nExternal class root: {EXTERNAL_CLASS_ROOT}\n'
    f'Original handle: {ORIGINAL_DATASET_HANDLE}\nOriginal resolved mount: {ORIGINAL_MOUNT}\n'
    f'Original download method: {ORIGINAL_LOAD_METHOD}\nOriginal six-class folders available: {ORIGINAL_AVAILABLE}\n',
    encoding='utf-8')


Existing Kaggle inputs: ['/kaggle/input/datasets']
Using existing Kaggle input [sumn2u/garbage-classification-v2]: /kaggle/input/datasets/sumn2u/garbage-classification-v2
Using existing Kaggle input [saumyamohandas/garbage-classification-image-dataset]: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset
External image-class root: /kaggle/input/datasets/sumn2u/garbage-classification-v2/original
Original Stage 2 root: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset
Original split/class folder found for every class: True


596

In [5]:
# 3 — Actual image counts for all 10 V2 classes; retain 6 only

def image_files(folder):
    if folder is None or not folder.is_dir():
        return []
    return sorted((p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in EXTS), key=str)

external_paths = OrderedDict((cl, image_files(EXTERNAL_DIRS[cl])) for cl in ALL)
external_counts = {cl:len(paths) for cl,paths in external_paths.items()}
original_paths = {}
for cl in KEEP:
    files = []
    for folder in original_directories[cl]:
        files.extend(image_files(folder))
    original_paths[cl] = list(dict.fromkeys(files))

if not all(external_counts[cl] > 0 for cl in KEEP):
    raise ValueError('At least one required external class is empty; check the attached Garbage V2 data.')
ORIGINAL_AVAILABLE = bool(ORIGINAL_AVAILABLE and all(original_paths[cl] for cl in KEEP))
all_count = sum(external_counts.values())
selected_count = sum(external_counts[cl] for cl in KEEP)
excluded_count = sum(external_counts[cl] for cl in EXCLUDE)
assert selected_count + excluded_count == all_count

print(f'{"External class":<16}{"Images":>8}   Decision')
print('-'*48)
for cl in ALL:
    print(f'{cl.title():<16}{external_counts[cl]:>8}   {"KEEP" if cl in KEEP else "EXCLUDE"}')
print('-'*48)
print('Total external images:',all_count)
print('Six-class CANDIDATE images:',selected_count)
print('Four-class excluded images:',excluded_count)
print('Original reference images available:',sum(map(len,original_paths.values())))

with (OUTPUT/'Section1_Ten_Class_Counts.csv').open('w',newline='',encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['external_class','external_image_count','section1_selection'])
    for cl in ALL:
        writer.writerow([cl,external_counts[cl],'KEEP_SIX_CLASS' if cl in KEEP else 'EXCLUDE_EXTRA_CLASS'])


External class    Images   Decision
------------------------------------------------
Battery              756   EXCLUDE
Biological           699   EXCLUDE
Cardboard           1411   KEEP
Clothes             1892   EXCLUDE
Glass               1736   KEEP
Metal                930   KEEP
Paper               1336   KEEP
Plastic             1597   KEEP
Shoes               1449   EXCLUDE
Trash                453   KEEP
------------------------------------------------
Total external images: 12259
Six-class CANDIDATE images: 7463
Four-class excluded images: 4796
Original reference images available: 2972


### Visual comparison (all six classes, with extra Trash coverage)

The notebook creates six side-by-side comparison sheets when the original reference dataset is accessible. It also creates contact sheets for **all external Trash images**, and original Trash images when available. Missing originals remain visibly marked; they are **not** replaced with guesses.


In [6]:
# 4 — Side-by-side comparison sheets and full Trash galleries (no model predictions)
SHEET_DIR = OUTPUT / 'Section1_Visual_Comparison'
SHEET_DIR.mkdir(parents=True, exist_ok=True)

try:
    FONT = ImageFont.load_default()
except Exception:
    FONT = None

def select_examples(paths, count, seed=2026):
    paths = list(paths)
    if len(paths) <= count:
        return paths
    picks = random.Random(seed).sample(range(len(paths)), count)
    return [paths[i] for i in sorted(picks)]

def draw_tile(canvas, p, x, y, tile_w=204, tile_h=166):
    draw = ImageDraw.Draw(canvas)
    try:
        with Image.open(p) as img:
            img = ImageOps.exif_transpose(img).convert('RGB')
            img.thumbnail((tile_w - 12, tile_h - 39), Image.Resampling.LANCZOS)
            bx = x + (tile_w - img.width) // 2
            by = y + 4 + (tile_h - 39 - img.height)//2
            canvas.paste(img, (bx,by))
    except (OSError, ValueError, Image.DecompressionBombError):
        draw.text((x+8,y+30), 'Image unreadable', fill=(130,0,0), font=FONT)
    caption = p.name if len(p.name) < 26 else p.name[:23] + '...'
    draw.text((x+5, y+tile_h-31), caption, fill=(0,0,0), font=FONT)

def compare_sheet(cls, original, external, per_side=12):
    original = select_examples(original, per_side, seed=2026)
    external = select_examples(external, per_side, seed=2026)
    rows = max(math.ceil(len(original)/4), math.ceil(len(external)/4))
    tile_w, tile_h = 204, 166
    width = 8 * tile_w + 16
    height = 56 + rows * tile_h
    canvas = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(canvas)
    draw.text((12,8), f'Stage 2 ORIGINAL: {cls.upper()}', fill=(0,0,0), font=FONT)
    draw.text((4*tile_w+12,8), f'Garbage V2 EXTERNAL: {cls.upper()}', fill=(0,0,0), font=FONT)
    draw.line((4*tile_w+8, 0, 4*tile_w+8,height), fill=(150,150,150), width=3)
    for j, p in enumerate(original):
        draw_tile(canvas,p, 8+(j%4)*tile_w, 48+(j//4)*tile_h)
    for j, p in enumerate(external):
        draw_tile(canvas,p, 8+4*tile_w+(j%4)*tile_w, 48+(j//4)*tile_h)
    output = SHEET_DIR / f'Section1_Compare_{cls.title()}.png'
    canvas.save(output)
    print('Saved comparison:', output.name)
    return output

if not ORIGINAL_AVAILABLE:
    print('WARNING: Original images unavailable; comparison sheets will show an empty original side. Do not mark semantic compatibility confirmed.')

comparison_files = {}
for cl in KEEP:
    comparison_files[cl] = compare_sheet(cl, original_paths[cl], external_paths[cl], per_side=12)

def all_images_contact_sheets(label, paths, per_sheet=24):
    paths = list(paths)
    gallery = SHEET_DIR / f'All_{label.replace(" ", "_")}'
    gallery.mkdir(exist_ok=True)
    paths_file = gallery/'filenames.csv'
    with paths_file.open('w',newline='',encoding='utf-8') as f:
        w = csv.writer(f); w.writerow(['sheet_number','slot_number','filename','full_path'])
        for j, p in enumerate(paths):
            w.writerow([j//per_sheet+1,j%per_sheet+1,p.name,str(p)])
    created = []
    for begin in range(0, len(paths), per_sheet):
        current = paths[begin:begin+per_sheet]
        tiles_w, tiles_h = 6, 4
        tile_w, tile_h = 204, 166
        canvas = Image.new('RGB', (tiles_w*tile_w, 46+tiles_h*tile_h), 'white')
        draw = ImageDraw.Draw(canvas)
        index = begin//per_sheet+1
        draw.text((8,12), f'{label}: page {index}/{math.ceil(len(paths)/per_sheet)}', fill=(0,0,0), font=FONT)
        for j,p in enumerate(current):
            draw_tile(canvas,p,(j%tiles_w)*tile_w,46+(j//tiles_w)*tile_h)
        dest=gallery/f'{label.replace(" ", "_")}_{index:03d}.jpg'
        canvas.save(dest,quality=86)
        created.append(dest)
    print(f'{label}: {len(paths)} images displayed across {len(created)} contact sheets')
    return created

external_trash_gallery = all_images_contact_sheets('V2_Trash', external_paths['trash'])
original_trash_gallery = all_images_contact_sheets('Stage2_Trash', original_paths['trash']) if ORIGINAL_AVAILABLE else []

print('Visual outputs:', SHEET_DIR)
print('Open the six comparison PNG files and Trash gallery JPGs in Kaggle Output.')


Saved comparison: Section1_Compare_Cardboard.png
Saved comparison: Section1_Compare_Glass.png
Saved comparison: Section1_Compare_Metal.png
Saved comparison: Section1_Compare_Paper.png
Saved comparison: Section1_Compare_Plastic.png
Saved comparison: Section1_Compare_Trash.png
V2_Trash: 453 images displayed across 19 contact sheets
Stage2_Trash: 349 images displayed across 15 contact sheets
Visual outputs: /kaggle/working/REMNET_Stage3_Section1_Both_Datasets/Section1_Visual_Comparison
Open the six comparison PNG files and Trash gallery JPGs in Kaggle Output.


## Automatic semantic analysis — no manual image upload

This step uses **CLIP ViT-B/32** to compare the images from the *original reference* and *external V2* datasets inside the Kaggle kernel. Every external Trash photograph is processed (not merely a thumbnail sample); each other external class and each original class is represented by a fixed-seed sample. Each image is compared with the six class descriptions and with real original-reference image embeddings. Ambiguous cases and class-definition shifts are flagged.

**Pretrained CLIP is an independent audit instrument**, not a substitute for human annotation or an estimate of REMNET accuracy. No images are excluded by this Section 1 code.

In [7]:
# 5 — Configure the independently pretrained semantic auditor (CPU only)
import numpy as np
import importlib.util
from collections import Counter

# We audit all external Trash images; the other classes use reproducible samples.
ORIGINAL_NONTRASH_SAMPLE_PER_CLASS = 100
EXTERNAL_NONTRASH_SAMPLE_PER_CLASS = 120
CLIP_MODEL_ID = 'openai/clip-vit-base-patch32'
CLIP_BATCH_SIZE = 12
AUDIT_SEED = 20261105

CLASS_PROMPTS = {
    'cardboard': [
        'a photograph of a discarded cardboard box or corrugated cardboard packaging',
        'an image of cardboard waste for recycling',
        'a photo of brown cardboard material',
    ],
    'glass': [
        'a photograph of discarded glass bottles or glass jars',
        'an image of glass waste for recycling',
        'a photograph of broken or intact glass packaging',
    ],
    'metal': [
        'a photograph of metal cans, aluminum foil or scrap metal waste',
        'an image of discarded metallic packaging',
        'a photograph of used aluminum beverage cans',
    ],
    'paper': [
        'a photograph of newspaper, paper sheets or paper waste',
        'an image of discarded loose paper for recycling',
        'a photograph of crumpled paper',
    ],
    'plastic': [
        'a photograph of plastic bottles, plastic containers or plastic packaging waste',
        'an image of discarded plastic material',
        'a photograph of plastic waste for recycling',
    ],
    'trash': [
        'a photograph of general residual trash, mixed garbage or litter',
        'an image of dirty mixed household waste or non-recyclable rubbish',
        'a photograph of miscellaneous discarded garbage that does not clearly fit one recyclable material category',
    ],
}

rng = random.Random(AUDIT_SEED)
def fixed_sample(paths, limit, full=False):
    paths=list(paths)
    if full or len(paths)<=limit:
        return paths
    return [paths[i] for i in sorted(rng.sample(range(len(paths)),limit))]

original_audit_paths = {c:fixed_sample(original_paths[c], ORIGINAL_NONTRASH_SAMPLE_PER_CLASS,full=(c=='trash')) for c in KEEP}
external_audit_paths = {c:fixed_sample(external_paths[c], EXTERNAL_NONTRASH_SAMPLE_PER_CLASS,full=(c=='trash')) for c in KEEP}
print('Audit sample sizes (original | V2):')
for cl in KEEP:
    print(f'  {cl:10s} {len(original_audit_paths[cl]):4d} | {len(external_audit_paths[cl]):4d}')
assert all(original_audit_paths[c] for c in KEEP), 'Original public dataset must provide all six classes for comparative semantic auditing.'
assert len(external_audit_paths['trash'])==len(external_paths['trash']), 'All external Trash images must be audited.'
print('Analysis device: CPU (CUDA not used)')


Audit sample sizes (original | V2):
  cardboard   100 |  120
  glass       100 |  120
  metal       100 |  120
  paper       100 |  120
  plastic     100 |  120
  trash       349 |  453
Analysis device: CPU (CUDA not used)


In [8]:
# 6 — Load pretrained CLIP safely on CPU (this is NOT REMNET inference)
import torch
try:
    torch.set_num_threads(max(1, min(4, torch.get_num_threads())))
except Exception:
    pass

def load_pretrained_clip():
    try:
        from transformers import CLIPModel, CLIPProcessor
    except Exception:
        print('Installing a compatible Hugging Face Transformers release...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.45,<5'])
        importlib.invalidate_caches()
        from transformers import CLIPModel, CLIPProcessor
    model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to('cpu').eval()
    processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
    return model,processor

CLIP_READY = False
CLIP_LOAD_ERROR = None
try:
    clip_model, clip_processor = load_pretrained_clip()
    CLIP_READY=True
    print('Independent pretrained visual auditor loaded:', CLIP_MODEL_ID, '| device: CPU')
except Exception as exc:
    CLIP_LOAD_ERROR = f'{type(exc).__name__}: {exc}'
    print('WARNING: CLIP model unavailable; automatic semantic audit is blocked.')
    print('Cause:', CLIP_LOAD_ERROR[:1000])
    print('Kaggle recovery: enable Internet in Settings for pretrained model download, then rerun this and later cells.')
    (OUTPUT/'Section1_CLIP_Load_Diagnostic.txt').write_text(CLIP_LOAD_ERROR+'\n',encoding='utf-8')


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

... (3 outputs left out)

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Independent pretrained visual auditor loaded: openai/clip-vit-base-patch32 | device: CPU


In [9]:
# 7 — Compute CLIP embeddings for BOTH datasets and all external Trash images
# This cell runs in Kaggle; it does not access any model checkpoint from REMNET.

def unit_rows(a):
    a=np.asarray(a,dtype=np.float32)
    return a/np.maximum(np.linalg.norm(a,axis=1,keepdims=True),1e-10)

def unwrap_features(value):
    if torch.is_tensor(value):return value
    # Compatible with differing Transformers versions.
    if getattr(value,'pooler_output',None) is not None:return value.pooler_output
    if isinstance(value,(tuple,list)) and value:return value[0]
    raise TypeError(f'Unsupported CLIP feature output type: {type(value)}')

original_records = [{'source':'Stage2_public_reference','class':cl,'path':p} for cl in KEEP for p in original_audit_paths[cl]]
external_records = [{'source':'GarbageV2','class':cl,'path':p} for cl in KEEP for p in external_audit_paths[cl]]

embedding_rows=[]
failed_images=[]
if CLIP_READY:
    all_records=original_records+external_records
    for start in range(0,len(all_records),CLIP_BATCH_SIZE):
        batch_recs=all_records[start:start+CLIP_BATCH_SIZE]
        imgs=[]; valid=[]
        for rec in batch_recs:
            try:
                with Image.open(rec['path']) as im:
                    imgs.append(ImageOps.exif_transpose(im).convert('RGB').copy())
                valid.append(rec)
            except Exception as exc:
                failed_images.append({'source':rec['source'],'class':rec['class'],'path':str(rec['path']),'error':str(exc)})
        if valid:
            with torch.inference_mode():
                tensors=clip_processor(images=imgs,return_tensors='pt')
                features=unwrap_features(clip_model.get_image_features(pixel_values=tensors['pixel_values']))
                feats=unit_rows(features.detach().cpu().numpy())
            for rec,emb in zip(valid,feats):
                embedding_rows.append((rec,emb))
        if start==0 or (start//CLIP_BATCH_SIZE)%15==0 or start+CLIP_BATCH_SIZE>=len(all_records):
            print('Images processed:',min(start+CLIP_BATCH_SIZE,len(all_records)),'/',len(all_records))
    if failed_images:
        with (OUTPUT/'Section1_Unreadable_Semantic_Audit_Images.csv').open('w',newline='',encoding='utf-8') as f:
            w=csv.DictWriter(f,fieldnames=['source','class','path','error']);w.writeheader();w.writerows(failed_images)
    # Do not silently analyze incomplete galleries as though complete.
    audited_trash_count=sum(rec['class']=='trash' and rec['source']=='GarbageV2' for rec,emb in embedding_rows)
    if audited_trash_count!=len(external_paths['trash']):
        print(f'WARNING: Only {audited_trash_count}/{len(external_paths["trash"])} Trash images were readable and scored.')
    print('Image embeddings complete:',len(embedding_rows),'/',len(all_records))
else:
    print('Automatic image-embedding audit skipped: CLIP model did not load.')


Images processed: 12 / 1902
Images processed: 192 / 1902
Images processed: 372 / 1902
Images processed: 552 / 1902
Images processed: 732 / 1902
Images processed: 912 / 1902
Images processed: 1092 / 1902
Images processed: 1272 / 1902
Images processed: 1452 / 1902
Images processed: 1632 / 1902
Images processed: 1812 / 1902
Images processed: 1902 / 1902
Image embeddings complete: 1902 / 1902


In [10]:
# 8 — Independent semantic compatibility analysis and per-image Trash flags
# A score is a screening signal, NOT a ground-truth class label.

def jensen_shannon(p,q):
    p=np.asarray(p,dtype=np.float64);q=np.asarray(q,dtype=np.float64)
    p=p/(p.sum()+1e-12);q=q/(q.sum()+1e-12)
    m=.5*(p+q)
    def kl(a,b):
        mask=a>0
        return np.sum(a[mask]*np.log2(a[mask]/np.maximum(b[mask],1e-15)))
    return float(np.sqrt(max(0,.5*kl(p,m)+.5*kl(q,m))))

semantic_rows=[];trash_flags=[];automatic_summary=[]
SEMANTIC_AUDIT_COMPLETE=False

if CLIP_READY and embedding_rows:
    recs=[r for r,v in embedding_rows]
    EMB=np.stack([v for r,v in embedding_rows])
    oi=np.array([i for i,r in enumerate(recs) if r['source']=='Stage2_public_reference'],dtype=int)
    ei=np.array([i for i,r in enumerate(recs) if r['source']=='GarbageV2'],dtype=int)
    OR=EMB[oi]; EX=EMB[ei]
    orec=[recs[i] for i in oi];erec=[recs[i] for i in ei]
    original_labels=np.array([r['class'] for r in orec]); external_labels=np.array([r['class'] for r in erec])
    # CLIP text embeddings averaged over several non-exclusive descriptions per class.
    txt_proto=[]
    with torch.inference_mode():
        for cl in KEEP:
            tok=clip_processor(text=CLASS_PROMPTS[cl],padding=True,truncation=True,return_tensors='pt')
            feat=unwrap_features(clip_model.get_text_features(input_ids=tok['input_ids'],attention_mask=tok['attention_mask']))
            txt_proto.append(unit_rows(unit_rows(feat.detach().cpu().numpy()).mean(axis=0,keepdims=True))[0])
    TXT=np.stack(txt_proto)
    orig_text_scores=OR@TXT.T
    ext_text_scores=EX@TXT.T
    orig_text_arg=np.argmax(orig_text_scores,axis=1)
    ext_text_arg=np.argmax(ext_text_scores,axis=1)
    # Compare each external picture with real original labeled pictures of all six classes.
    sim=EX@OR.T
    orig_pred_scores=np.empty((len(EX),len(KEEP)),dtype=np.float32)
    for j,cl in enumerate(KEEP):
        scores=sim[:,original_labels==cl]
        k=min(3,scores.shape[1]);orig_pred_scores[:,j]=np.sort(scores,axis=1)[:,-k:].mean(axis=1)
    orig_ref_arg=np.argmax(orig_pred_scores,axis=1)
    class_counts_by_source={}
    for cl in KEEP:
        j=KEEP.index(cl)
        m_orig=(original_labels==cl)
        m_ext=(external_labels==cl)
        n_orig=int(m_orig.sum()); n_ext=int(m_ext.sum())
        if n_orig<3 or n_ext<3:
            status='REVIEW_REQUIRED'
            js=float('nan'); orig_label_rate=float('nan'); ext_label_rate=float('nan'); ref_match=float('nan'); flagged_fraction=1.
        else:
            oh=np.bincount(orig_text_arg[m_orig],minlength=6)
            eh=np.bincount(ext_text_arg[m_ext],minlength=6)
            js=jensen_shannon(oh,eh)
            orig_label_rate=float((orig_text_arg[m_orig]==j).mean())
            ext_label_rate=float((ext_text_arg[m_ext]==j).mean())
            ref_match=float((orig_ref_arg[m_ext]==j).mean())
            flagged=0
            for idx in np.flatnonzero(m_ext):
                rank=np.argsort(ext_text_scores[idx])
                best=int(rank[-1]);second=int(rank[-2]); ref_best=int(orig_ref_arg[idx])
                text_margin=float(ext_text_scores[idx,best]-ext_text_scores[idx,second])
                ref_sort=np.sort(orig_pred_scores[idx]);ref_margin=float(ref_sort[-1]-ref_sort[-2])
                # Flag only when BOTH independent signals confidently prefer some other class.
                strong_disagreement=(best!=j and ref_best!=j and text_margin>=.012 and ref_margin>=.014)
                if strong_disagreement:flagged+=1
                semantic_rows.append({
                    'external_class':cl,
                    'image_path':str(erec[idx]['path']),
                    'image_filename':erec[idx]['path'].name,
                    'clip_top_category':KEEP[best],
                    'clip_true_category_score':round(float(ext_text_scores[idx,j]),5),
                    'clip_top_category_score':round(float(ext_text_scores[idx,best]),5),
                    'clip_top_vs_second_margin':round(text_margin,5),
                    'reference_nearest_category':KEEP[ref_best],
                    'reference_true_category_score':round(float(orig_pred_scores[idx,j]),5),
                    'reference_top_vs_second_margin':round(ref_margin,5),
                    'auto_flag':'REVIEW_CANDIDATE' if strong_disagreement else 'NO_STRONG_CONFLICT',
                    'audit_model':CLIP_MODEL_ID,
                })
                if cl=='trash' and strong_disagreement:
                    trash_flags.append(semantic_rows[-1])
            flagged_fraction=flagged/n_ext
            # Compare the original-versus-external VISUAL class distributions.
            # The threshold is a review heuristic, not a statistical test or certification.
            status=('AUTO_SUPPORTED' if js <= .32 and
                    ext_label_rate >= max(.15,orig_label_rate-.25) and
                    ref_match >= .45 and flagged_fraction <= .20
                    else 'REVIEW_REQUIRED')
        automatic_summary.append({
            'class':cl,
            'original_reference_images_analyzed':n_orig,
            'external_images_analyzed':n_ext,
            'external_all_images_count':int(external_counts[cl]),
            'external_audit_coverage':round(n_ext/max(1,external_counts[cl]),5),
            'orig_clip_label_alignment_fraction':round(orig_label_rate,4),
            'external_clip_label_alignment_fraction':round(ext_label_rate,4),
            'external_original_reference_nearest_class_fraction':round(ref_match,4),
            'original_vs_external_clip_label_js_distance':round(js,4),
            'confident_dual_signal_conflict_fraction':round(flagged_fraction,4),
            'automatic_semantic_assessment':status,
            'interpretation':'Model-assisted screening; ambiguous category labels require independent human confirmation',
        })
    SEMANTIC_AUDIT_COMPLETE=(not failed_images and len(semantic_rows)==len(external_records) and
          len(original_records)==len(oi) and len(external_records)==len(ei))
    print('\nAUTOMATIC CLASS-LEVEL COMPATIBILITY ASSESSMENT')
    for row in automatic_summary:
        print(f"{row['class'].title():10s} {row['automatic_semantic_assessment']:16s} "
              f"original={row['original_reference_images_analyzed']} external={row['external_images_analyzed']} "
              f"JS={row['original_vs_external_clip_label_js_distance']:.3f} "
              f"orig-ref-agreement={row['external_original_reference_nearest_class_fraction']:.3f}")
    print('External Trash pictures audited:',sum(x['class']=='trash' for x in erec),'/',len(external_paths['trash']))
    print('Trash pictures flagged for closer review:',len(trash_flags))
    print('Automated semantic screening complete:',SEMANTIC_AUDIT_COMPLETE)
else:
    print('No independent CLIP semantic results available. No unsupported compatibility claim will be made.')



AUTOMATIC CLASS-LEVEL COMPATIBILITY ASSESSMENT
Cardboard  REVIEW_REQUIRED  original=100 external=120 JS=0.365 orig-ref-agreement=0.442
Glass      AUTO_SUPPORTED   original=100 external=120 JS=0.151 orig-ref-agreement=0.867
Metal      AUTO_SUPPORTED   original=100 external=120 JS=0.120 orig-ref-agreement=0.808
Paper      AUTO_SUPPORTED   original=100 external=120 JS=0.158 orig-ref-agreement=0.792
Plastic    AUTO_SUPPORTED   original=100 external=120 JS=0.103 orig-ref-agreement=0.767
Trash      AUTO_SUPPORTED   original=349 external=453 JS=0.152 orig-ref-agreement=0.815
External Trash pictures audited: 453 / 453
Trash pictures flagged for closer review: 4
Automated semantic screening complete: True


In [11]:
# 9 — Save interpretable scientific evidence, screening results and Trash images flagged
import zipfile

with (OUTPUT/'Section1_Automated_Class_Compatibility.csv').open('w',newline='',encoding='utf-8') as f:
    cols=['class','original_reference_images_analyzed','external_images_analyzed','external_all_images_count',
          'external_audit_coverage','orig_clip_label_alignment_fraction','external_clip_label_alignment_fraction',
          'external_original_reference_nearest_class_fraction','original_vs_external_clip_label_js_distance',
          'confident_dual_signal_conflict_fraction','automatic_semantic_assessment','interpretation']
    w=csv.DictWriter(f,fieldnames=cols);w.writeheader();w.writerows(automatic_summary)

with (OUTPUT/'Section1_Per_Image_Automated_Semantic_Flags.csv').open('w',newline='',encoding='utf-8') as f:
    cols=['external_class','image_path','image_filename','clip_top_category','clip_true_category_score',
          'clip_top_category_score','clip_top_vs_second_margin','reference_nearest_category',
          'reference_true_category_score','reference_top_vs_second_margin','auto_flag','audit_model']
    w=csv.DictWriter(f,fieldnames=cols);w.writeheader();w.writerows(semantic_rows)

with (OUTPUT/'Section1_Trash_Automatic_Review_Candidates.csv').open('w',newline='',encoding='utf-8') as f:
    w=csv.DictWriter(f,fieldnames=cols);w.writeheader();w.writerows(trash_flags)

# Additional visual sheets of flagged external Trash images; no labels are changed/excluded.
FLAG_DIR=OUTPUT/'Section1_Automatically_Flagged_Trash'
FLAG_DIR.mkdir(exist_ok=True)
flag_paths=[Path(r['image_path']) for r in trash_flags]

def flagged_contact_sheets(paths, prefix, per_page=30):
    saved=[]
    for i in range(0,len(paths),per_page):
        group=paths[i:i+per_page]
        ncol=5;nrow=int(math.ceil(len(group)/ncol))
        out=Image.new('RGB',(ncol*215,nrow*170),(245,245,245))
        for j,p in enumerate(group):
            draw_tile(out,p,(j%ncol)*215,(j//ncol)*170,215,170)
        name=FLAG_DIR/f'{prefix}_{i//per_page+1:03d}.jpg'
        out.save(name,quality=88)
        saved.append(name)
    return saved
if flag_paths:
    flagged_contact_sheets(flag_paths,'Flagged_Trash')

all_supported=bool(SEMANTIC_AUDIT_COMPLETE and automatic_summary) and all(r['automatic_semantic_assessment']=='AUTO_SUPPORTED' for r in automatic_summary)
status={
  'stage':'Stage 3 Section 1 only',
  'original_dataset_url':'https://www.kaggle.com/datasets/saumyamohandas/garbage-classification-image-dataset',
  'external_dataset_url':'https://www.kaggle.com/datasets/sumn2u/garbage-classification-v2',
  'external_kaggle_version_requested':EXTERNAL_DATASET_VERSION,
  'external_version_evidence':EXTERNAL_VERSION_ACTUAL,
  'external_all_image_count':int(all_count),
  'external_selected_six_image_count':int(selected_count),
  'external_excluded_four_image_count':int(excluded_count),
  'original_public_reference_image_file_count':sum(len(x) for x in original_paths.values()),
  'original_public_count_not_stage2_clean_split':True,
  'semantic_analysis_method':'independent pretrained CLIP zero-shot text and original-image-reference similarity',
  'model_id':CLIP_MODEL_ID,
  'device':'cpu',
  'semantic_model_loaded':bool(CLIP_READY),
  'model_error':CLIP_LOAD_ERROR,
  'semantic_automatic_analysis_complete':bool(SEMANTIC_AUDIT_COMPLETE),
  'all_six_automatically_supported':bool(all_supported),
  'external_trash_images_audited':sum(1 for rec in semantic_rows if rec['external_class']=='trash'),
  'external_trash_images_total':int(external_counts['trash']),
  'trash_auto_flagged_for_review':len(trash_flags),
  'human_verified_semantic_compatibility':False,
  'section1_review_complete':False,
  'qualification':'This is a model-assisted semantic SCREEN; it is not proof of ground-truth compatibility or an image-level annotation audit. No overlap removal is performed.',
}
(OUTPUT/'Section1_Automated_Semantic_Review_Summary.json').write_text(json.dumps(status,indent=2),encoding='utf-8')

readme=('REMNET STAGE 3 SECTION 1 — AUTOMATED SEMANTIC AUDIT\n'
        'Both datasets are loaded in Kaggle and compared by an independent pretrained model.\n'
        'Automatic flags are candidate label inconsistencies, not confirmed annotation errors.\n'
        'No REMNET inference, no model training, no duplicate removal or pHash is performed.\n'
        'Only selected representative images from the five non-Trash external classes are analyzed.\n'
        'All 453 external Trash images are processed when readable and model is available.\n'
        'The public original reference dataset is not necessarily byte-identical to the 2906 retained Stage 2 images.\n'
        'Human-reviewed compatibility remains separate from automated screening.\n')
(OUTPUT/'Section1_Automated_Semantic_Review_README.txt').write_text(readme,encoding='utf-8')
archive=OUTPUT/'Section1_Automatic_Semantic_Audit_Package.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(OUTPUT.rglob('*')):
        if p.is_file() and p!=archive and 'extracted_datasets' not in p.parts:
            z.write(p,p.relative_to(OUTPUT))

print('\nSECTION 1 — AUTOMATED AUDIT RESULT')
print('Original public reference images:',status['original_public_reference_image_file_count'])
print('External 10-class images:',all_count)
print('Six selected candidate images:',selected_count)
print('Extra four-class images excluded:',excluded_count)
print('Semantic automatic analysis complete:',status['semantic_automatic_analysis_complete'])
print('External Trash images evaluated by pretrained audit model:',status['external_trash_images_audited'],'/',status['external_trash_images_total'])
print('Trash review candidates:',len(trash_flags))
print('Final outputs:',archive)



SECTION 1 — AUTOMATED AUDIT RESULT
Original public reference images: 2972
External 10-class images: 12259
Six selected candidate images: 7463
Extra four-class images excluded: 4796
Semantic automatic analysis complete: True
External Trash images evaluated by pretrained audit model: 453 / 453
Trash review candidates: 4
Final outputs: /kaggle/working/REMNET_Stage3_Section1_Both_Datasets/Section1_Automatic_Semantic_Audit_Package.zip


### Interpretation of these results

- `Section1_Automated_Class_Compatibility.csv` compares all six class distributions **using visual evidence from both Kaggle datasets**, with the number of images actually inspected for each class.
- `Section1_Per_Image_Automated_Semantic_Flags.csv` records automated predictions and original-reference similarity for each analyzed V2 image, including **all external Trash photographs**.
- `Section1_Trash_Automatic_Review_Candidates.csv` contains suspicious Trash examples **for additional independent scrutiny**; its entries are **not confirmed errors** and are not automatically excluded.
- `semantic_automatic_analysis_complete: true` means the independent AI screen executed for its planned image sample. It **does not** mean the manual/annotation-level review has been completed.
- The six-class count of 7,463 is **before** SHA-256/pHash removal and must not be reported as the final independent validation count.

If the pretrained CLIP download is blocked by Kaggle settings, the notebook will still save class counts and comparison images, but automated semantics will remain incomplete rather than producing invented conclusions.